In [5]:
from preprocesamiento import preprocesar_texto
from nltk.util import ngrams
from collections import Counter, defaultdict
import math
import pandas as pd

In [28]:
def crear_modelo_trigramas(corpus: str) -> dict:
    """
    Crea un modelo de lenguaje basado en trigramas.
    
    Args:
        corpus: texto completo
        
    Returns:
        modelo: diccionario con probabilidades P(w3 | w1, w2)
        tokens: lista de tokens procesados
    """
    tokens = preprocesar_texto(corpus)
    
    # Agregar tokens de inicio y fin de oración
    tokens = ['<s>', '<s>'] + tokens + ['</s>']
    
    # Contar bigramas y trigramas
    bigramas = list(ngrams(tokens, 2))
    trigramas = list(ngrams(tokens, 3))
    
    bigrama_counts = Counter(bigramas)
    trigrama_counts = Counter(trigramas)
    
    # Modelo: P(w3 | w1, w2) = count(w1, w2, w3) / count(w1, w2)
    modelo = defaultdict(lambda: defaultdict(float))
    for (w1, w2, w3), count in trigrama_counts.items():
        modelo[(w1, w2)][w3] = count / bigrama_counts[(w1, w2)]
    
    return modelo, tokens, bigrama_counts, trigrama_counts

In [33]:
def calcular_entropia_perplejidad_trigramas(modelo, tokens):
    """
    Calcula la entropía y perplejidad de un modelo de trigramas.
    """
    trigramas = list(ngrams(tokens, 3))
    N = len(trigramas)
    
    log_prob_sum = 0
    for w1, w2, w3 in trigramas:
        prob = modelo[(w1, w2)].get(w3, 1e-6)
        log_prob_sum += math.log2(prob)
    
    entropia = - (1 / N) * log_prob_sum
    if entropia == 0:
        entropia = abs(entropia)
    perplejidad = 2 ** entropia if entropia >= 0 else float('inf')
    
    return entropia, perplejidad

In [4]:
def prob_sentence(sentence: str, modelo: dict, method='normal', k=0.01):
    """
    Calcula la probabilidad de una oración con un modelo de trigramas.
    
    Args:
        sentence: oración a evaluar
        modelo: dict con probabilidades P(w3 | w1, w2)
        method: 'normal', 'laplace' o 'add_k'
        k: constante para add-k smoothing
    
    Returns:
        probabilidad total de la oración
    """
    words = preprocesar_texto(sentence)
    words = ['<s>', '<s>'] + words + ['</s>']
    prob = 1.0
    vocabulario = set([w3 for pares in modelo.values() for w3 in pares])  # conjunto de palabras posibles
    V = len(vocabulario)
    
    for i in range(len(words) - 2):
        w1, w2, w3 = words[i], words[i + 1], words[i + 2]
        
        if method == 'normal':
            prob *= modelo[(w1, w2)].get(w3, 1e-6)  # prob normal o valor pequeño si no existe
            
        elif method == 'laplace':
            numerador = modelo[(w1, w2)].get(w3, 0) * (V) + 1
            denominador = V + 1
            prob *= numerador / denominador
            
        elif method == 'add_k':
            numerador = modelo[(w1, w2)].get(w3, 0) * (V) + k
            denominador = V + k * V
            prob *= numerador / denominador
            
    return prob

# Ejemplo

In [7]:
news = pd.read_csv('D57000_complete.csv', sep=';')

In [12]:
news

,ID,Label,Titulo,Descripcion,Fecha
0,ID,1,Moreno intenta apaciguar el flanco sanitario m...,El presidente abre la puerta a unos comicios e...,19/04/2022
1,ID,1,La Abogacía del Estado se retira como acusació...,"En un escrito, la abogada del Estado Rosa Marí...",17/09/2021
2,ID,0,Las promesas incumplidas de Pablo Echenique en...,Este lunes y martes la Asamblea de Madrid acog...,12/09/2022
3,ID,1,Sánchez defiende 'resolver el problema' de la ...,Resulta evidente que la ley ha tenido algunos ...,07/02/2023
4,ID,1,Ian Gibson cierra la lista electoral de la con...,"El hispanista, que ya ocupó un puesto simbólic...",12/04/2023
...,...,...,...,...,...
57226,ID,0,Mónica García: 'La ultraderecha ya no nos seña...,El líder de PP abandona el debate de la Cadena...,23/04/2021
57227,ID,1,Sigue aquí la comparecencia de Luis Bárcenas e...,Luis Bárcenas comparece en la comisión del Con...,05/05/2020
57228,ID,0,Albiol recupera la alcaldía de Badalona tras n...,Desde Guanyem rechazaron dividirse a partes ig...,12/05/2020
57229,ID,1,El mando de la Guardia Civil que reconoció tor...,Exteriores revela que el teniente coronel Pedr...,16/03/2021


In [81]:
test_corpus = ''
for i in range(10):
    test_corpus += news.loc[i]['Titulo'] + '\n' + news.loc[i]['Descripcion']+ '\n'
test_corpus

"Moreno intenta apaciguar el flanco sanitario mientras enreda con la fecha de las elecciones\nEl presidente abre la puerta a unos comicios en junio que no sean en domingo.\nLa Abogacía del Estado se retira como acusación en la pieza de Iberdrola del 'caso Tándem'\nEn un escrito, la abogada del Estado Rosa María Seoane argumenta su decisión en la falta de legitimación activa respecto de los delitos imputados.\nLas promesas incumplidas de Pablo Echenique en sanidad, educación y vivienda\nEste lunes y martes la Asamblea de Madrid acogerá el debate del estado de la región. El último se celebró hace dos años, el 14 de septiembre de 2020.\nSánchez defiende 'resolver el problema' de la ley del 'solo sí es sí' desde el diálogo\nResulta evidente que la ley ha tenido algunos efectos indeseados. Y me quedo corto, ha destacado el presidente del Gobierno ante sus diputados y senadores.\nIan Gibson cierra la lista electoral de la confluencia de Podemos en Granada\nEl hispanista, que ya ocupó un pues

In [82]:
modelo_eje, tokens_eje, ejem_bigrama_counts, ejem_trigrama_counts = crear_modelo_trigramas(test_corpus)

In [83]:
entropia_eje, perplejidad_eje = calcular_entropia_perplejidad_trigramas(modelo_eje, tokens_eje)
print(f'Entropía: {entropia_eje}, Perplejidad: {perplejidad_eje}')

Entropía: 0.008130081300813009, Perplejidad: 1.005651251345443


In [84]:
# encontrar bigramas con conteo mayor a 1
bigrama_mayor_1 = {k: v for k, v in ejem_bigrama_counts.items() if v > 1}
bigrama_mayor_1

{('eleccion', 'municipal'): 2}

In [53]:
frase_prueba = 'Se acercan las fechas de las elecciones del presidente Moreno. Mientras tanto, la abogacía del estado se retira como acusación por promesas incumplidas de Pablo Echenique.'
print(f'Probabilidad de trigramas sin suavizado: {prob_sentence(frase_prueba, modelo_eje, method="normal")}')
print(f'\nProbabilidad de trigramas con suavizado Laplace: {prob_sentence(frase_prueba, modelo_eje, method="laplace")}')
print(f'\nProbabilidad de trigramas con suavizado Add-k: {prob_sentence(frase_prueba, modelo_eje, method="add_k", k=0.01)}')


Probabilidad de trigramas sin suavizado: 9.999999999999997e-61

Probabilidad de trigramas con suavizado Laplace: 3.765071185835269e-24

Probabilidad de trigramas con suavizado Add-k: 3.428774627805213e-44


## Análisis

Debido a que únicamente se han utilizado 10 noticias y sus detalles para la creación del modelo, la entropía es cercana a 0 y la perplejidad es cercana a 1. Esto se debe a que las noticias no están relacionadas entre sí y algunas son reales y otras, falsas. Esto causa que no haya más conteos para todos los trigramas y bigramas; todo es completamente aleatorio. Si se utilizaran las 57,000 noticias, esto cambiaría. Por la misma razón, al evaluar una oración 'prueba', aunque se usen palabras dentro de estas noticias, su probabilidad sigue siendo extremadamente baja.